<a href="https://colab.research.google.com/github/isa-ulisboa/greends-pml/blob/main/notebooks/T10_CNN_Grape_Disease_Identification_Yolo8n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00


Import package and load a pre-trained model for image classification

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load a model
model = YOLO("yolov8n-cls.pt")  # load a pretrained model (recommended for training)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Indicate where the data is and fine tune the model

In [ ]:
# Train the model
data_folder=Path('/content/drive/MyDrive/INV_PROJ/FORSAID/grape_desease')
results = model.train(data=data_folder, epochs=3, imgsz=224, batch=32, verbose=True)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/INV_PROJ/FORSAID, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

You're asking a great question that gets to the heart of how convolutional layers work! Let's break down the `[3, 16, 3, 2]` arguments for `layer 0` and how they relate to your input `(C, W, H, B) = (3, 224, 224, 32)`.

For a `ultralytics.nn.modules.conv.Conv` layer, the arguments typically follow this order:

*   `input_channels`: The number of channels in the input feature map to this layer.
*   `output_channels`: The number of feature maps (channels) this layer will produce.
*   `kernel_size`: The size of the convolutional filter (e.g., `3` means a 3x3 filter).
*   `stride`: How many pixels the kernel shifts at each step (e.g., `2` means it skips every other pixel, effectively downsampling).

Now, let's map `[3, 16, 3, 2]` to your input `(C, W, H, B)`:

1.  **`input_channels = 3`**: This directly corresponds to your `C=3` (Channels). Your input images have 3 color channels (Red, Green, Blue). The first convolutional layer expects this 3-channel input.

2.  **`output_channels = 16`**: This means the convolutional layer will apply 16 different filters (kernels) to the input. Each filter learns to detect a different feature (e.g., edges, textures). The output of this layer will have 16 channels, regardless of the input channel count.

3.  **`kernel_size = 3`**: This indicates that the convolutional filters are 3x3 pixels. These small filters slide across your image, performing dot products to extract features.

4.  **`stride = 2`**: This is crucial for how `W=224` and `H=224` are processed. A stride of 2 means the filter moves 2 pixels at a time. This has the effect of downsampling the spatial dimensions (width and height) of the input. For an input of `224x224`, after a `3x3` convolution with `stride=2` (and typically some padding, which is often implicit or calculated to maintain certain sizes), the output spatial dimensions would be approximately halved. For example, `224 / 2 = 112`. So, the output of layer 0 would have dimensions like `(16, 112, 112)` (channels, height, width).

### What about `B=32` (Batch Size)?

As explained previously, the batch size `B=32` is a dimension that's handled at a higher level by the deep learning framework (PyTorch in this case, used by Ultralytics). When you define a layer like `Conv`, you're describing how it transforms a *single* input (e.g., one image with its channels, width, and height). The framework then automatically applies this transformation to all 32 images in your batch simultaneously. So, the layer definition itself focuses on the per-item transformation, not the batch dimension.

You're looking at the `ultralytics.nn.modules.block.C2f` layer, which is a key component in YOLOv8's backbone and neck for efficient feature extraction. The `C2f` block (Cross-Stage Partial-attention Fusion) is an improved version of the C3 block found in YOLOv5, designed to enhance performance and efficiency.

The arguments `[32, 32, 1, True]` correspond to the following parameters for the `C2f` module:

*   **`32` (Input Channels)**: This is the number of input channels (feature maps) that the `C2f` block receives from the previous layer. In your architecture, layer 2's `C2f` block receives input from layer 1, which has `32` output channels.

*   **`32` (Output Channels)**: This is the number of output channels that the `C2f` block will produce. It determines the depth of the feature map after processing within this block. In this case, the number of channels remains the same as the input.

*   **`1` (Number of Bottlenecks/Blocks)**: This parameter specifies how many internal bottleneck blocks are used within the `C2f` structure. A bottleneck block typically reduces and then restores the number of channels to efficiently learn features. Here, `1` indicates a relatively compact `C2f` block.

*   **`True` (Shortcut/Residual Connection)**: This boolean argument controls whether a shortcut connection (also known as a residual connection) is used within the `C2f` block. When `True`, it means that the input to the block is added to its output at some point, which helps in training deeper networks by mitigating the vanishing gradient problem and allowing for easier flow of information. This is a common practice in modern convolutional neural networks.